# Reproducing Nakamae et al. 2025 — CBE RNA off-target risk prediction

**Paper:** Nakamae K, Suzuki T, Yonezawa S, Yamamoto K, Kakuzaki T, Ono H,
Naito Y, Bono H. *Risk Prediction of RNA Off-Targets of CRISPR Base Editors in
Tissue-Specific Transcriptomes Using Language Models.* Int J Mol Sci 2025;
26(4):1723. [doi:10.3390/ijms26041723](https://doi.org/10.3390/ijms26041723)

**Original code** (MIT): [PiCTURE](https://github.com/KazukiNakamae/PiCTURE),
[PROTECTiO](https://github.com/KazukiNakamae/PROTECTiO)
**Models:** [STL](https://huggingface.co/KazukiNakamae/STLmodel),
[SNL](https://huggingface.co/KazukiNakamae/SNLmodel)

This notebook is a reproduction, not original work. The pipeline, models and
data are the authors'. What is mine is the verification: re-running their
released analysis and checking whether the published claims fall out of it.

---

## What is and is not reproducible here

Be explicit about this, because "I reproduced the paper" means three very
different things depending on which tier you reached.

| Tier | What it covers | Feasible on Colab? |
|---|---|---|
| **1. Tissue-specific ESD analysis** (Fig. 5, Table claims about colon/lung vs brain/ovary) | the authors committed the aggregated per-tissue ESD data for all four classifiers | **Yes** — pandas + scipy, no GPU, minutes |
| **2. Classifier inference** (load STL/SNL, predict on sequences) | both fine-tuned models are public on HuggingFace | **Yes** — GPU helpful, not required |
| **3. Motif classifier logic** (ACW/WCW/NNN) | trivially reimplementable; check against their `pred_motif_*.py` | **Yes** |
| **4. Fine-tuning Tables 1 & 2** | needs the PiCTURE-derived train/eval/test FASTA splits | **Only if those splits are released** — checked below |
| **5. PiCTURE from raw RNA-seq** | 13 SRA datasets, STAR two-pass + GATK on hg38 | **No.** Hundreds of GB and days of compute. Do not attempt on Colab. |

Tier 5 is the honest limit. If anyone asks whether you reproduced the paper
end to end, the answer is no — you reproduced the downstream risk analysis and
the classifiers, starting from the authors' released intermediate data. That is
a normal and respectable scope for a reproduction, and claiming more would fall
apart on the first question about STAR runtimes.

---
## 1. Setup

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pandas", "scipy", "statsmodels", "matplotlib", "seaborn",
                "biopython"], check=True)

import os, glob, json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
print("ready")

In [ ]:
# Clone the authors' repository. MIT licensed, so re-use with attribution
# is fine -- but this notebook CALLS their code rather than copying it into
# a repo of my own, which keeps authorship unambiguous.
if not os.path.isdir("PROTECTiO"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KazukiNakamae/PROTECTiO.git"], check=True)

os.chdir("/content/PROTECTiO" if os.path.isdir("/content/PROTECTiO") else "PROTECTiO")
print("cwd:", os.getcwd())
print()
subprocess.run(["git", "log", "-1", "--format=commit %h  %ad  %s"], check=False)
print("\nRecord that commit hash. A reproduction without the commit it ran "
      "against is not reproducible itself.")

---
## 2. Tier 1 — what data did they actually release?

The repo ships directories named `aggregated_ESD_*` and `summary_ESD_*`, one
set per classifier. Before assuming anything about their contents, look.

In [ ]:
for d in sorted(glob.glob("aggregated_ESD_*")) + sorted(glob.glob("summary_ESD_*")):
    files = glob.glob(os.path.join(d, "**", "*"), recursive=True)
    files = [f for f in files if os.path.isfile(f)]
    total = sum(os.path.getsize(f) for f in files)
    print(f"{d}")
    print(f"   {len(files)} files, {total/1e6:.2f} MB")
    for f in sorted(files)[:6]:
        print(f"     {os.path.relpath(f, d)}  ({os.path.getsize(f)/1e3:.1f} KB)")
    if len(files) > 6:
        print(f"     ... and {len(files)-6} more")
    print()

In [ ]:
# Inspect the schema of whatever CSVs are there, rather than guessing.
# The classifier suffixes map to the paper as:
#   (no suffix) -> STL model     _snv1pred -> SNL model
#   _acwpred    -> ACW motif     _wcwpred  -> WCW motif
CLASSIFIERS = {
    "STL model":  "aggregated_ESD_v1_0_0_Human_RNA-seq_CBE",
    "SNL model":  "aggregated_ESD_v1_0_0_Human_RNA-seq_CBE_snv1pred",
    "ACW motif":  "aggregated_ESD_v1_0_0_Human_RNA-seq_CBE_acwpred",
    "WCW motif":  "aggregated_ESD_v1_0_0_Human_RNA-seq_CBE_wcwpred",
}

for name, d in CLASSIFIERS.items():
    csvs = sorted(glob.glob(os.path.join(d, "**", "*.csv"), recursive=True))
    print(f"=== {name}  ({d}) ===")
    if not csvs:
        print("   no CSVs found -- check the directory listing above\n")
        continue
    sample = csvs[0]
    df = pd.read_csv(sample)
    print(f"   {len(csvs)} CSV(s); example: {os.path.relpath(sample, d)}")
    print(f"   shape {df.shape}, columns: {list(df.columns)}")
    print(df.head(3).to_string(index=False))
    print()

---
## 3. Tier 1 — reproduce the tissue ESD comparison

ESD (effective substrate density) is, per the paper: the number of *effective*
potential risk substrates in a transcript — cytosines whose C-to-U conversion
would create a premature stop codon AND which the classifier calls positive —
normalised by the total number of PRSs and by the protein's amino-acid length.

The published claims to test:

- **brain, adipose, ovary, heart, muscle** — significantly *lower* ESD than all
  tissue-specific transcripts
- **prostate, colon, lung** — significantly *higher*
- **colon highest**, consistently across all four classifiers
- the direction holds for motif classifiers as well as the language models

The next cell loads each classifier's per-tissue ESD, computes the mean and a
two-tailed Welch's t-test against the "All" reference group, and checks each
claim. Welch rather than Student because the tissue groups have wildly
different n and no reason to share a variance.

In [ ]:
def load_tissue_esd(directory):
    """Load per-tissue ESD values from an aggregated_ESD_* directory.

    Structure is discovered rather than assumed: the function looks for a
    per-tissue CSV layout first, then a single long-format table. If neither
    matches it raises, which is correct -- a silent fallback here would
    produce a plausible-looking plot from the wrong column.
    """
    csvs = sorted(glob.glob(os.path.join(directory, "**", "*.csv"), recursive=True))
    if not csvs:
        raise FileNotFoundError(f"no CSVs under {directory}")

    frames = []

    for path in csvs:
        df = pd.read_csv(path)
        cols_lower = {c.lower(): c for c in df.columns}

        # find the ESD column
        esd_col = None
        for key in ("density", "esd", "effective_substrate_density"):
            hits = [c for lc, c in cols_lower.items() if key in lc]
            if hits:
                esd_col = hits[0]
                break
        if esd_col is None:
            continue

        # find or infer the tissue column
        tissue_col = None
        for key in ("tissue", "organ", "group"):
            hits = [c for lc, c in cols_lower.items() if key in lc]
            if hits:
                tissue_col = hits[0]
                break

        if tissue_col:
            sub = df[[tissue_col, esd_col]].copy()
            sub.columns = ["tissue", "esd"]
        else:
            # one file per tissue: take the tissue from the filename
            tissue = os.path.splitext(os.path.basename(path))[0]
            for junk in ("_density", "density_", "pred_dnabert2_cbe_snv1_",
                         "pred_motif_acw_", "pred_motif_wcw_", "_esd"):
                tissue = tissue.replace(junk, "")
            sub = pd.DataFrame({"tissue": tissue, "esd": df[esd_col]})

        frames.append(sub)

    if not frames:
        raise ValueError(f"found CSVs under {directory} but no ESD column in any")

    out = pd.concat(frames, ignore_index=True)
    out["esd"] = pd.to_numeric(out["esd"], errors="coerce")
    return out.dropna(subset=["esd"])


esd_data = {}
for name, d in CLASSIFIERS.items():
    try:
        esd_data[name] = load_tissue_esd(d)
        n_t = esd_data[name].tissue.nunique()
        print(f"{name:11s}  {len(esd_data[name]):>7,} transcripts, {n_t} tissue groups")
    except Exception as e:
        print(f"{name:11s}  FAILED: {type(e).__name__}: {e}")

if not esd_data:
    print("\nNothing loaded. Re-read the directory listing in section 2 and "
          "adjust load_tissue_esd to the actual schema before continuing. "
          "Do not proceed on partial data.")

In [ ]:
PAPER_LOWER = ["brain", "adipose", "ovary", "ovaries", "heart", "muscle"]
PAPER_HIGHER = ["prostate", "colon", "lung", "lungs"]

def norm(t):
    return str(t).strip().lower().replace(" ", "_")

def test_tissues(df, ref_labels=("all",)):
    """Mean ESD per tissue + Welch's t-test against the reference group."""
    df = df.copy()
    df["key"] = df.tissue.map(norm)

    ref = df[df.key.isin([norm(r) for r in ref_labels])].esd.values
    if len(ref) == 0:
        # No explicit "All" row: use the pooled set as the reference, and say so.
        ref = df.esd.values
        ref_note = "pooled (no explicit 'All' group found)"
    else:
        ref_note = "explicit 'All' group"

    rows = []
    for tissue, grp in df.groupby("tissue"):
        if norm(tissue) in [norm(r) for r in ref_labels]:
            continue
        vals = grp.esd.values
        if len(vals) < 3:
            continue
        t, p = stats.ttest_ind(vals, ref, equal_var=False)
        rows.append({"tissue": tissue, "n": len(vals),
                     "mean_esd": vals.mean(), "sem": stats.sem(vals),
                     "t": t, "p": p,
                     "direction": "higher" if vals.mean() > ref.mean() else "lower",
                     "significant": p < 0.05})
    return pd.DataFrame(rows).sort_values("mean_esd"), ref.mean(), ref_note


summary = {}
for name, df in esd_data.items():
    res, ref_mean, ref_note = test_tissues(df)
    summary[name] = res
    print(f"\n{'='*70}\n{name}   (reference: {ref_note}, mean ESD {ref_mean:.5f})\n{'='*70}")
    print(res[["tissue", "n", "mean_esd", "p", "direction", "significant"]]
          .to_string(index=False))

In [ ]:
print("="*70)
print("CLAIM CHECK vs the published Figure 5")
print("="*70)

for name, res in summary.items():
    print(f"\n--- {name} ---")
    res = res.copy()
    res["key"] = res.tissue.map(norm)

    def check(group, expected):
        hits = res[res.key.apply(lambda k: any(g in k for g in [norm(group)]))]
        if hits.empty:
            return f"  {group:10s} not present in this dataset"
        r = hits.iloc[0]
        ok = (r.direction == expected) and r.significant
        mark = "OK  " if ok else "MISS"
        return (f"  {mark} {group:10s} expected {expected:6s} | "
                f"got {r.direction:6s} mean {r.mean_esd:.5f} p={r.p:.2e} "
                f"{'sig' if r.significant else 'ns'}")

    for g in ["brain", "ovar", "heart", "muscle", "adipose"]:
        print(check(g, "lower"))
    for g in ["colon", "lung", "prostate"]:
        print(check(g, "higher"))

    # The paper's strongest single claim: colon is the highest-risk tissue.
    if not res.empty:
        top = res.sort_values("mean_esd", ascending=False).iloc[0]
        print(f"  highest-ESD tissue here: {top.tissue} ({top.mean_esd:.5f})")
        print(f"    paper says colon should be notably highest -> "
              f"{'consistent' if 'colon' in norm(top.tissue) else 'DIFFERS'}")

print("\n" + "="*70)
print("A MISS is not necessarily an error in the paper. Check in this order:")
print("  1. did load_tissue_esd pick the right ESD column?")
print("  2. is the reference group the same one they used ('All' = transcripts")
print("     tissue-specific in at least one tissue, up OR down)?")
print("  3. has the repo data changed since publication? (commit hash above)")
print("Report what you found either way -- a reproduction that silently drops")
print("its disagreements is not a reproduction.")

In [ ]:
n = len(summary)
if n:
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5), squeeze=False)
    for ax, (name, res) in zip(axes[0], summary.items()):
        r = res.sort_values("mean_esd")
        colors = ["#4A7B9D" if d == "lower" else "#B85042" for d in r.direction]
        ax.barh(r.tissue, r.mean_esd, xerr=r.sem, color=colors, height=0.7)
        for i, (_, row) in enumerate(r.iterrows()):
            if row.significant:
                ax.text(row.mean_esd + row.sem, i, " *", va="center", fontsize=11)
        ax.set_xlabel("mean ESD")
        ax.set_title(name, fontsize=11)
        ax.tick_params(labelsize=8)
    plt.tight_layout()
    plt.savefig("reproduced_figure5_tissue_esd.png", dpi=150, bbox_inches="tight")
    plt.show()

    out = pd.concat([r.assign(classifier=k) for k, r in summary.items()],
                    ignore_index=True)
    out.to_csv("reproduced_tissue_esd_stats.csv", index=False)
    print("saved reproduced_figure5_tissue_esd.png and "
          "reproduced_tissue_esd_stats.csv")
    print("\n* marks p < 0.05, Welch's two-tailed, vs the reference group.")

---
## 4. Tier 3 — reimplement the motif classifiers and check them

The ACW/WCW classifiers are the paper's baselines, and they are simple enough
to write from the description alone. Doing so independently and then diffing
against the authors' `pred_motif_acw.py` / `pred_motif_wcw.py` is the cheapest
real verification in this whole notebook: if my reading of "ACW motif at the
substitution site" matches their code's behaviour, the baseline comparison in
Tables 1 and 2 is interpretable. If it doesn't, something about the position
convention is different and every motif number needs re-reading.

Per the methods, SNL inputs are 40 bp: 20 bp upstream + the substrate C +
19 bp downstream. So the C sits at **index 20** (0-based) and the motif is
`seq[19:22]`.

In [ ]:
print(open("pred_motif_acw.py").read())

In [ ]:
print(open("pred_motif_wcw.py").read())

In [ ]:
# My independent implementation, from the paper text only.
W = {"A", "T", "U"}
C_INDEX = 20          # 20 bp upstream => substrate C at 0-based index 20

def motif_at_site(seq, c_index=C_INDEX):
    s = seq.strip().upper().replace("U", "T")
    if len(s) < c_index + 2:
        return None
    return s[c_index - 1: c_index + 2]      # [5' neighbour, C, 3' neighbour]

def predict_acw(seq):
    m = motif_at_site(seq)
    if m is None or m[1] != "C":
        return 0
    return int(m[0] == "A" and m[2] in W)

def predict_wcw(seq):
    m = motif_at_site(seq)
    if m is None or m[1] != "C":
        return 0
    return int(m[0] in W and m[2] in W)

def predict_nnn(seq):
    # The paper's "NNN" classifier: predicts positive for any site.
    # It exists to show what accuracy looks like with no information at all,
    # which is why its precision collapses (0.383 in Table 1, 0.25 in Table 2).
    return 1

# Sanity checks against the definitions
tests = [
    ("A" * 19 + "ACA" + "T" * 18, 1, 1, "ACA: ACW yes, WCW yes"),
    ("A" * 19 + "TCT" + "T" * 18, 0, 1, "TCT: ACW no, WCW yes"),
    ("A" * 19 + "GCG" + "T" * 18, 0, 0, "GCG: neither"),
    ("A" * 19 + "ACG" + "T" * 18, 0, 0, "ACG: 3' not W -> neither"),
    ("A" * 19 + "GCA" + "T" * 18, 0, 1, "GCA: WCW yes, ACW no"),
]
print(f"{'sequence motif':<16} {'ACW':>4} {'exp':>4} {'WCW':>4} {'exp':>4}   note")
ok = True
for seq, exp_acw, exp_wcw, note in tests:
    a, w = predict_acw(seq), predict_wcw(seq)
    flag = "" if (a == exp_acw and w == exp_wcw) else "   <-- MISMATCH"
    ok &= (a == exp_acw and w == exp_wcw)
    print(f"{motif_at_site(seq):<16} {a:>4} {exp_acw:>4} {w:>4} {exp_wcw:>4}   {note}{flag}")
print("\nall definition checks passed" if ok else "\nDEFINITION MISMATCH -- fix before using")
print("\nNow compare the printed source above: does their C index and motif")
print("window match C_INDEX=20 and seq[19:22]? If not, note the difference.")

---
## 5. Tier 2 — load the fine-tuned models and run them

Both models are public. This is the part people assume is hard and isn't — the
authors did the expensive fine-tuning and released the weights.

Requires `einops`. `trust_remote_code=True` is mandatory: DNABERT-2's
architecture ships in the model repo, not in `transformers`.

If you hit a triton or flash-attn import error, the usual fix is
`!pip install -q "transformers==4.29.2"` then restart the runtime.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "einops", "torch"], check=True)

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

MODELS = {"SNL": "KazukiNakamae/SNLmodel", "STL": "KazukiNakamae/STLmodel"}

loaded = {}
for tag, repo in MODELS.items():
    try:
        tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
        mdl = AutoModelForSequenceClassification.from_pretrained(
            repo, trust_remote_code=True).to(DEVICE).eval()
        loaded[tag] = (tok, mdl)
        print(f"{tag}: loaded, {sum(p.numel() for p in mdl.parameters())/1e6:.1f}M params")
    except Exception as e:
        print(f"{tag}: FAILED {type(e).__name__}: {e}")

In [ ]:
def predict_lm(tag, seqs, batch_size=32):
    tok, mdl = loaded[tag]
    probs = []
    for i in range(0, len(seqs), batch_size):
        batch = [s.strip().upper().replace("U", "T") for s in seqs[i:i+batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True)
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = mdl(**enc).logits
        probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.array(probs)


# Demo on the motif test sequences. This checks the models RUN and that their
# output has the expected shape -- it is NOT an accuracy measurement, since
# these five sequences are hand-made and unlabelled.
if loaded:
    demo = [t[0] for t in tests]
    rows = []
    for seq in demo:
        rows.append({"motif": motif_at_site(seq),
                     "ACW": predict_acw(seq), "WCW": predict_wcw(seq)})
    out = pd.DataFrame(rows)
    for tag in loaded:
        out[f"P({tag})"] = np.round(predict_lm(tag, demo), 3)
    print(out.to_string(index=False))
    print("\nSanity only. Five synthetic sequences measure nothing about accuracy.")

---
## 6. Tier 2 — run their standalone prediction on the example transcript

The repo ships `example/ENST00000288774_target_with_header.fasta` and a
`stand_alone_prediction.sh` wrapper. Running all four classifiers over the same
transcript is the most faithful small-scale reproduction available: identical
input, identical code, your execution.

This needs their conda environment, so it may not run cleanly on bare Colab.
If it fails, that is a finding worth recording — reproducibility includes
whether the shipped environment installs.

In [ ]:
print(os.path.exists("example/ENST00000288774_target_with_header.fasta"))
print(open("example/ENST00000288774_target_with_header.fasta").read()[:400])

In [ ]:
# Attempt the authors' wrapper for each classifier. Failures are captured
# rather than raised, because "which of these ran" is itself the result.
RUNS = {
    "ACW motif": "pred_motif_acw.py",
    "WCW motif": "pred_motif_wcw.py",
    "STL model": "pred_dnabert2_cbe_sv1.py",
    "SNL model": "pred_dnabert2_cbe_snv1.py",
}

run_log = {}
for name, script in RUNS.items():
    outdir = f"repro_standalone_{script.replace('.py','')}"
    cmd = ["bash", "stand_alone_prediction.sh",
           "-O", outdir, "-p", script,
           "-t", "example/ENST00000288774_target_with_header.fasta",
           "-i", "fasta"]
    print(f"\n=== {name} ===")
    r = subprocess.run(cmd, capture_output=True, text=True)
    run_log[name] = r.returncode
    print("returncode:", r.returncode)
    if r.stdout:
        print(r.stdout[-1500:])
    if r.returncode != 0 and r.stderr:
        print("STDERR:", r.stderr[-1500:])

print("\n" + "="*60)
print("summary:", {k: ("ok" if v == 0 else f"failed({v})") for k, v in run_log.items()})
print("If all four failed on environment issues, fall back to section 5, which")
print("calls the models directly and does not need their conda env.")

---
## 7. Tier 4 — are the fine-tuning splits available?

Tables 1 and 2 can only be reproduced with the exact train/eval/test FASTA
splits derived from PiCTURE. Check before claiming anything about those tables.

In [ ]:
if not os.path.isdir("../PiCTURE"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/KazukiNakamae/PiCTURE.git", "../PiCTURE"],
                   check=False)

pic = "../PiCTURE"
if os.path.isdir(pic):
    hits = []
    for root, dirs, files in os.walk(pic):
        dirs[:] = [d for d in dirs if d != ".git"]
        for f in files:
            if f.endswith((".fa", ".fasta", ".csv", ".tsv", ".txt")):
                p = os.path.join(root, f)
                hits.append((os.path.relpath(p, pic), os.path.getsize(p)))
    hits.sort(key=lambda x: -x[1])
    print(f"{len(hits)} data-like files in PiCTURE, largest first:\n")
    for p, s in hits[:25]:
        print(f"  {s/1e3:>10.1f} KB  {p}")

    keywords = ("positive", "negative", "train", "test", "eval", "40bp", "101bp")
    cands = [h for h in hits if any(k in h[0].lower() for k in keywords)]
    print(f"\nfiles matching split keywords: {len(cands)}")
    for p, s in cands[:20]:
        print(f"  {s/1e3:>10.1f} KB  {p}")

    if not cands:
        print("\nNo fine-tuning splits in the repo. Tables 1 and 2 are therefore")
        print("NOT reproducible from public files alone -- they require rerunning")
        print("PiCTURE on the 13 SRA datasets (Tier 5, not feasible here).")
        print("State this as a limitation rather than quoting their numbers as")
        print("if you had verified them.")

---
## 8. Published numbers, for reference only

These are transcribed from the paper so your reproduced values have something
to sit next to. **Do not present these as your results** — they are the
authors'. Only the ESD analysis in section 3 and the runs in sections 5–6 are
yours.

**Table 1 — STL model** (imbalanced splits, 101 bp, overlaps between sets)

| | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| STL model | 0.803 | 0.729 | 0.672 | 0.690 |
| ACW motif | 0.596 | 0.636 | 0.686 | 0.580 |
| WCW motif | 0.703 | 0.663 | 0.721 | 0.662 |
| NNN motif | 0.766 | 0.383 | 0.500 | 0.434 |

**Table 2 — SNL model** (balanced, de-duplicated, 40 bp)

| | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| SNL model | 0.726 | 0.729 | 0.726 | 0.725 |
| ACW motif | 0.670 | 0.697 | 0.670 | 0.659 |
| WCW motif | 0.717 | 0.717 | 0.717 | 0.716 |
| NNN motif | 0.500 | 0.250 | 0.500 | 0.333 |

### The two things worth noticing in these tables

**The NNN row is the tell.** In Table 1 the all-positive classifier gets
**0.766 accuracy** — higher than the WCW motif classifier and only 0.037 below
the STL model — purely because positives outnumber negatives 3.1:1. Its
precision is 0.383. That is the whole argument for why the authors rebuilt the
dataset balanced for Table 2, where NNN accuracy drops to 0.500 as it should.
If you take one methodological point from this paper, it is that: the authors
included a no-information baseline, and it exposed their first dataset.

**The SNL model's accuracy is *lower* than the STL model's** (0.726 vs 0.803),
and that is the better model. The numbers aren't comparable because the test
sets differ — balanced and de-duplicated versus imbalanced with overlaps
between train and test. A reader comparing 0.803 to 0.726 across the two tables
would draw exactly the wrong conclusion. On the harder, fairer evaluation the
SNL model beats WCW on every metric, which Table 1's model could not claim.

**The honest margin.** In Table 2 the SNL model beats the WCW motif baseline by
0.009 accuracy and 0.009 F1. Neither table reports a confidence interval. With
20,514 test sequences the standard error on accuracy is roughly 0.003, so the
gap is probably real but small — a 117M-parameter language model buying about a
percentage point over a three-nucleotide string match. The authors' stronger
and better-supported claim is the one about precision on canonical off-targets
(Figure 3): motif classifiers averaged below 0.3 while the language models
exceeded 0.5, meaning motif-only screening would flag a lot of sites that
aren't real.

---
## 9. Write up what you got

```python
# run this last
```

In [ ]:
print("REPRODUCTION SUMMARY")
print("="*60)
print()
print("Tier 1 (tissue ESD):       ", "DONE" if summary else "not completed")
print("Tier 2 (load STL/SNL):     ", "DONE" if loaded else "not completed")
print("Tier 2 (their wrapper):    ",
      {k: ('ok' if v == 0 else 'failed') for k, v in run_log.items()} if 'run_log' in dir() else "not attempted")
print("Tier 3 (motif reimpl):     ", "DONE" if ok else "definition mismatch")
print("Tier 4 (Tables 1-2):       ", "see section 7")
print("Tier 5 (PiCTURE from SRA): ", "NOT ATTEMPTED - infeasible on Colab")
print()
print("Checklist before you write this up anywhere:")
print("  [ ] PROTECTiO commit hash recorded (section 1)")
print("  [ ] every claim-check MISS in section 3 explained or flagged")
print("  [ ] Nakamae et al. 2025 cited, MIT license of their code noted")
print("  [ ] the published tables labelled as theirs, not yours")
print("  [ ] Tier 5 limitation stated explicitly")
print()
print("Files to keep:")
for f in ["reproduced_figure5_tissue_esd.png", "reproduced_tissue_esd_stats.csv"]:
    print(f"  {f}: {'exists' if os.path.exists(f) else 'MISSING'}")